# Depth & Surface Normal Extraction — Proof of Concept

This notebook implements the foundation of our first-principles pipeline:
1. Take a 2D image as input
2. Extract raw Z-depth using the ZoeDepth AI model
3. Calculate surface normal vectors (gradients) from that depth data

## Setup
- **Runtime**: Go to `Runtime > Change runtime type` and select **T4 GPU**
- Run **Cell 1** to install dependencies, then **Cell 2** to execute the pipeline

In [ ]:
# Cell 1: Install the Dependencies
# Before we can run the AI models, we need to install the required
# PyTorch and computer vision libraries.

!pip install timm
!pip install torch torchvision
!pip install matplotlib pillow numpy scipy

In [ ]:
# Cell 2: The Core Extraction Engine
# This script downloads a sample image, pulls the ZoeDepth AI model into
# the GPU, extracts the metric depth, and mathematically derives the
# surface normals using the Z-buffer gradients.

import torch
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import urllib.request

# 1. Load a sample image (you can later replace this with your own uploaded file)
print("Downloading sample image...")
url = "https://raw.githubusercontent.com/isl-org/ZoeDepth/main/assets/demo.png"
image_path = "sample_input.png"
urllib.request.urlretrieve(url, image_path)
img = Image.open(image_path).convert("RGB")

# 2. Initialize the ZoeDepth AI Model from PyTorch Hub
print("Loading ZoeDepth AI model into GPU...")
repo = "isl-org/ZoeDepth"
model_zoe_n = torch.hub.load(repo, "ZoeD_N", pretrained=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_zoe_n.eval().to(device)

# 3. Extract the Z-Depth Map
print("Extracting volumetric depth...")
depth_numpy = model_zoe_n.infer_pil(img)

# Normalize the depth to a 0.0 - 1.0 range for math operations
depth_min, depth_max = depth_numpy.min(), depth_numpy.max()
depth_normalized = (depth_numpy - depth_min) / (depth_max - depth_min)

# 4. Compute Surface Normals using First Principles (Gradients)
print("Calculating surface normals from the depth gradient...")
# We use NumPy to find the rate of change (slope) along the X and Y axes
dzdx = np.gradient(depth_normalized, axis=1)
dzdy = np.gradient(depth_normalized, axis=0)

# Construct the normal vectors [-dz/dx, -dz/dy, 1]
# A multiplier can be added here later to exaggerate the relief depth
normal_map = np.dstack((-dzdx, -dzdy, np.ones_like(depth_normalized)))

# Normalize the vectors so their length equals 1
norm = np.linalg.norm(normal_map, axis=2, keepdims=True)
normal_map_normalized = normal_map / norm

# Convert vector math (-1 to 1) into RGB color space (0 to 1) for visualization
normal_vis = (normal_map_normalized + 1.0) / 2.0

# 5. Render the outputs
print("Rendering visualization...")
fig, axs = plt.subplots(1, 3, figsize=(18, 6))

axs[0].imshow(img)
axs[0].set_title("1. Original Image")
axs[0].axis('off')

axs[1].imshow(depth_normalized, cmap='inferno')
axs[1].set_title("2. AI Extracted Global Depth")
axs[1].axis('off')

axs[2].imshow(normal_vis)
axs[2].set_title("3. Mathematically Derived Surface Normals")
axs[2].axis('off')

plt.tight_layout()
plt.show()

## How This Achieves Our Baseline

- **Global Volume (Panel 2):** ZoeDepth gives us an accurate estimation of the overall
  scene depth. However, the depth map is too soft for direct CNC machining or 3D printing —
  it lacks the sharp, coin-like detail required for high-end bas-relief.

- **Local Surface Data (Panel 3):** By taking the derivative (gradients `dzdx` and `dzdy`)
  of the AI's depth map, we generate a high-frequency normal map. The varying colors
  represent the exact physical angles light would hit the surface.

## Next Step

Take these two maps and run them through a proprietary compression algorithm that
flattens the depth (Panel 2) while multiplying and hard-coding the local gradients
(Panel 3) back onto that flattened plane.